# Project: HNSW Performance Benchmarking
Now that you’ve seen how HNSW parameters and payload indexes affect performance with the DBpedia dataset, it’s time to optimize for your own domain and use case.

## Your Mission
Build on your Day 1 search engine by adding performance optimization. You’ll discover which HNSW settings work best for your specific data and queries, and measure the real impact of payload indexing.

## Estimated Time: 90 minutes

What You’ll Build
A performance-optimized version of your Day 1 search engine that demonstrates:

- Fast bulk load: Load with m=0, then switch to HNSW
- HNSW parameter tuning: Try different m and ef_construct
- Payload indexing impact: Time filtering with and without indexes
- Domain findings: What works best for your content

## Setup
Prerequisites

- Qdrant Cloud cluster (URL + API key)
- Python 3.9+ (or Google Colab)
- Packages: qdrant-client, sentence-transformers, numpy

Models
- Embeddings: sentence-transformers/all-MiniLM-L6-v2 (384-dim)

Dataset

Reuse your Day 1 domain data or prepare a dataset with 1,000+ items and a rich text field (e.g., description).
Include a few numeric fields for filtering (e.g., length, word_count) so payload indexing impact can be measured.

In [50]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [17]:
from movie_recommendation.dataset import MoviesDataset

dataset = MoviesDataset('MOVIES_API')

dataset

In [20]:
movies = dataset.get_enhanced_latest(size=1000)

Fetching latest movies - starting page: 1, total size: 1000
Fetching page 1...
Got 20 movies from page 1
Fetching page 2...
Got 20 movies from page 2
Fetching page 3...
Got 20 movies from page 3
Fetching page 4...
Got 20 movies from page 4
Fetching page 5...
Got 20 movies from page 5
Fetching page 6...
Got 20 movies from page 6
Fetching page 7...
Got 20 movies from page 7
Fetching page 8...
Got 20 movies from page 8
Fetching page 9...
Got 20 movies from page 9
Fetching page 10...
Got 20 movies from page 10
Fetching page 11...
Got 20 movies from page 11
Fetching page 12...
Got 20 movies from page 12
Fetching page 13...
Got 20 movies from page 13
Fetching page 14...
Got 20 movies from page 14
Fetching page 15...
Got 20 movies from page 15
Fetching page 16...
Got 20 movies from page 16
Fetching page 17...
Got 20 movies from page 17
Fetching page 18...
Got 20 movies from page 18
Fetching page 19...
Got 20 movies from page 19
Fetching page 20...
Got 20 movies from page 20
Fetching page 21..

In [21]:
[movie.title for movie in movies][:10]

['Obsession',
 "Lee Cronin's The Mummy",
 'Kara',
 "Tom Clancy's Jack Ryan: Ghost War",
 'Backrooms',
 'The Punisher: One Last Kill',
 'Project Hail Mary',
 'The Super Mario Galaxy Movie',
 'Dhurandhar: The Revenge',
 'Swapped']

In [6]:
from movie_recommendation.search import HNSWConfig


configs: list[HNSWConfig] = [
    HNSWConfig(name='fast_initial_upload', connection_per_node=0, built_graph_accuracy=100),
    HNSWConfig(name='memory_optimized', connection_per_node=8, built_graph_accuracy=100),
    HNSWConfig(name='balanced', connection_per_node=16, built_graph_accuracy=200),
    HNSWConfig(name='high_quality', connection_per_node=32, built_graph_accuracy=400), 
]

configs

[HNSWConfig(name='fast_initial_upload', connection_per_node=0, built_graph_accuracy=100),
 HNSWConfig(name='memory_optimized', connection_per_node=8, built_graph_accuracy=100),
 HNSWConfig(name='balanced', connection_per_node=16, built_graph_accuracy=200),
 HNSWConfig(name='high_quality', connection_per_node=32, built_graph_accuracy=400)]

In [22]:
from movie_recommendation import MovieSearch
from movie_recommendation.point_builder import encoders, vector_configs

movie_search_collections: list[MovieSearch] = []


for config in configs:
    collection_name = f'movie_recommedation-{config.name}'
    movie_search = MovieSearch(
        encoders, 
        vector_configs,
        collection_name=collection_name,
        hnsw_config=config
    )
    movie_search_collections.append(movie_search)


for idx, movie_search in enumerate(movie_search_collections):
    print(f"====Creating {idx} collection {movie_search.collection_name}")
    was_created = movie_search.create_collection()

====Creating 0 collection movie_recommedation-fast_initial_upload
Collection 'movie_recommedation-fast_initial_upload' already exists
Created collection 'movie_recommedation-fast_initial_upload'
====Creating 1 collection movie_recommedation-memory_optimized
Collection 'movie_recommedation-memory_optimized' already exists
Created collection 'movie_recommedation-memory_optimized'
====Creating 2 collection movie_recommedation-balanced
Collection 'movie_recommedation-balanced' already exists
Created collection 'movie_recommedation-balanced'
====Creating 3 collection movie_recommedation-high_quality
Collection 'movie_recommedation-high_quality' already exists
Created collection 'movie_recommedation-high_quality'


In [23]:
for collection in movie_search_collections:
    print("==== Collection Config =====")
    print(collection.client.get_collection(collection.collection_name))

==== Collection Config =====
status=<CollectionStatus.GREEN: 'green'> optimizer_status=<OptimizersStatusOneOf.OK: 'ok'> warnings=None indexed_vectors_count=0 points_count=0 segments_count=2 config=CollectionConfig(params=CollectionParams(vectors={'fixed': VectorParams(size=384, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None), 'poster': VectorParams(size=512, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None), 'semantic': VectorParams(size=768, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None)}, shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, read_fan_out_delay_ms=None, on_disk_payload=True, sparse_vectors=None), hnsw_config=HnswConfig(m=100, ef_construct=100, full_sc

In [26]:
from movie_recommendation import PointBuilder
from pathlib import Path
import os

# Set up paths for saving
cache_dir = Path("data/movie_vectors")
cache_dir.mkdir(parents=True, exist_ok=True)

# Check if we have cached points
cached_file = cache_dir / "movie_points_1000.pkl"
incremental_file = cache_dir / "movie_points_incremental.pkl"

point_builder = PointBuilder()

# Try to load existing points first
if cached_file.exists():
    print(f"Loading cached points from {cached_file}")
    movie_points = point_builder.load_points_from_disk(cached_file)
else:
    print("Building points from movies...")
    movie_points = []
    
    # Process in batches to enable incremental saving
    batch_size = 50  # Save every 50 movies
    total_movies = len(movies)
    
    for i in range(0, total_movies, batch_size):
        batch_end = min(i + batch_size, total_movies)
        batch_movies = movies[i:batch_end]
        
        print(f"\nProcessing movies {i+1}-{batch_end} of {total_movies}")
        
        # Build points for this batch
        batch_points = point_builder.build_points_from_movies(batch_movies)
        movie_points.extend(batch_points)
        
        # Save incrementally
        print(f"Saving {len(movie_points)} points incrementally...")
        point_builder.save_points_to_disk(movie_points, incremental_file)
        
        # Also save a summary for debugging
        summary_file = cache_dir / f"movie_points_summary_{batch_end}.json"
        point_builder.save_points_as_json(movie_points, summary_file)
        
        print(f"✓ Saved up to movie {batch_end}/{total_movies}")
    
    # Final save with proper filename
    print(f"\nSaving final {len(movie_points)} points to {cached_file}")
    point_builder.save_points_to_disk(movie_points, cached_file)
    
    # Clean up incremental file
    if incremental_file.exists():
        incremental_file.unlink()
        print("Cleaned up incremental file")

print(f"\nTotal points: {len(movie_points)}")
movie_points[:10]

Building points from movies...

Processing movies 1-50 of 1000
Building points for 50 movies with batch_size=32, max_workers=4
  Processed 50/50 images
Successfully encoded 50 images
Encoding text chunks in batches...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11133.79it/s]


  Encoding 198 fixed chunks in batches...
  Encoding 490 semantic chunks in batches...
Created 936 points total
Saving 936 points incrementally...
Saving 936 points to data/movie_vectors/movie_points_incremental.pkl
Successfully saved points to data/movie_vectors/movie_points_incremental.pkl
Saving 936 points to data/movie_vectors/movie_points_summary_50.json as JSON
Successfully saved points summary to data/movie_vectors/movie_points_summary_50.json
✓ Saved up to movie 50/1000

Processing movies 51-100 of 1000
Building points for 50 movies with batch_size=32, max_workers=4
  Processed 50/50 images
Successfully encoded 50 images
Encoding text chunks in batches...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11669.10it/s]


  Encoding 176 fixed chunks in batches...
  Encoding 495 semantic chunks in batches...
Created 907 points total
Saving 1843 points incrementally...
Saving 1843 points to data/movie_vectors/movie_points_incremental.pkl
Successfully saved points to data/movie_vectors/movie_points_incremental.pkl
Saving 1843 points to data/movie_vectors/movie_points_summary_100.json as JSON
Successfully saved points summary to data/movie_vectors/movie_points_summary_100.json
✓ Saved up to movie 100/1000

Processing movies 101-150 of 1000
Building points for 50 movies with batch_size=32, max_workers=4
  Processed 50/50 images
Successfully encoded 50 images
Encoding text chunks in batches...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9591.34it/s]


  Encoding 207 fixed chunks in batches...
  Encoding 609 semantic chunks in batches...
Created 1102 points total
Saving 2945 points incrementally...
Saving 2945 points to data/movie_vectors/movie_points_incremental.pkl
Successfully saved points to data/movie_vectors/movie_points_incremental.pkl
Saving 2945 points to data/movie_vectors/movie_points_summary_150.json as JSON
Successfully saved points summary to data/movie_vectors/movie_points_summary_150.json
✓ Saved up to movie 150/1000

Processing movies 151-200 of 1000
Building points for 50 movies with batch_size=32, max_workers=4
  Processed 50/50 images
Successfully encoded 50 images
Encoding text chunks in batches...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7886.76it/s]
'[Errno 8] nodename nor servname provided, or not known' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/adapter_config.json
Retrying in 1s [Retry 1/5].


RuntimeError: Cannot send a request, as the client has been closed.

In [35]:
movies_points = point_builder.load_points_from_disk('./data/movie_vectors/movie_points_incremental.pkl')

Loading points from data/movie_vectors/movie_points_incremental.pkl
Successfully loaded 10971 points from data/movie_vectors/movie_points_incremental.pkl


In [36]:
for collection in movie_search_collections:
    if collection.collection_name == 'movie_recommedation-fast_initial_upload':
        continue
    print(f"==== Upload points to collection {collection.collection_name} ==== ")
    collection.upload_points(movie_points)

==== Upload points to collection movie_recommedation-memory_optimized ==== 
Uploaded batch 1/110
Uploaded batch 2/110
Uploaded batch 3/110
Uploaded batch 4/110
Uploaded batch 5/110
Uploaded batch 6/110
Uploaded batch 7/110
Uploaded batch 8/110
Uploaded batch 9/110
Uploaded batch 10/110
Uploaded batch 11/110
Uploaded batch 12/110
Uploaded batch 13/110
Uploaded batch 14/110
Uploaded batch 15/110
Uploaded batch 16/110
Uploaded batch 17/110
Uploaded batch 18/110
Uploaded batch 19/110
Uploaded batch 20/110
Uploaded batch 21/110
Uploaded batch 22/110
Uploaded batch 23/110
Uploaded batch 24/110
Uploaded batch 25/110
Uploaded batch 26/110
Uploaded batch 27/110
Uploaded batch 28/110
Uploaded batch 29/110
Uploaded batch 30/110
Uploaded batch 31/110
Uploaded batch 32/110
Uploaded batch 33/110
Uploaded batch 34/110
Uploaded batch 35/110
Uploaded batch 36/110
Uploaded batch 37/110
Uploaded batch 38/110
Uploaded batch 39/110
Uploaded batch 40/110
Uploaded batch 41/110
Uploaded batch 42/110
Uploaded 

In [37]:
import time

from qdrant_client import models

def wait_for_indexing(collection: MovieSearch, timeout=60, poll_interval=1):
    print(f"Waiting for collection '{collection.collection_name}' to be indexed...")
    start_time = time.time()

    while time.time() - start_time < timeout:
        info = collection.client.get_collection(collection_name)

        if info.indexed_vectors_count and info.indexed_vectors_count > 0 and info.status == models.CollectionStatus.GREEN:
            print(f"Success! Collection '{collection_name}' is indexed and ready.")
            print(f" - Status: {info.status.value}")
            print(f" - Indexed vectors: {info.indexed_vectors_count}")
            return

        print(f" - Status: {info.status.value}, Indexed vectors: {info.indexed_vectors_count}. Waiting...")
        time.sleep(poll_interval)

    info = collection.client.get_collection(collection_name)
    raise Exception(
        f"Timeout reached after {timeout} seconds. Collection '{collection_name}' is not ready. "
        f"Final status: {info.status.value}, Indexed vectors: {info.indexed_vectors_count}"
    )

collections = { collection.collection_name: collection  for collection in movie_search_collections }


for config in configs:
    if config.connection_per_node > 0:  # m=0 has no HNSW to wait for
        collection_name = f'movie_recommedation-{config.name}'
        wait_for_indexing(collections[collection_name])

Waiting for collection 'movie_recommedation-memory_optimized' to be indexed...
Success! Collection 'movie_recommedation-memory_optimized' is indexed and ready.
 - Status: green
 - Indexed vectors: 24230
Waiting for collection 'movie_recommedation-balanced' to be indexed...
Success! Collection 'movie_recommedation-balanced' is indexed and ready.
 - Status: green
 - Indexed vectors: 24230
Waiting for collection 'movie_recommedation-high_quality' to be indexed...
 - Status: yellow, Indexed vectors: 24230. Waiting...
 - Status: yellow, Indexed vectors: 24230. Waiting...
 - Status: yellow, Indexed vectors: 24230. Waiting...
 - Status: yellow, Indexed vectors: 24230. Waiting...
 - Status: yellow, Indexed vectors: 24230. Waiting...
 - Status: yellow, Indexed vectors: 24230. Waiting...
 - Status: yellow, Indexed vectors: 24230. Waiting...
 - Status: yellow, Indexed vectors: 24230. Waiting...
 - Status: yellow, Indexed vectors: 24230. Waiting...
 - Status: yellow, Indexed vectors: 24230. Waitin

In [38]:
test_query = 'american movie'

performance_results = {}

for config in configs:
    if config.connection_per_node > 0:
        collection_name = f'movie_recommedation-{config.name}'
        performance_results[collection_name] = (
            collections[collection_name].benchmark_search(test_query)
        )

===Start search fo the 64 value===
Search 0th time for 64
Search 1th time for 64
Search 2th time for 64
Search 3th time for 64
Search 4th time for 64
Search 5th time for 64
Search 6th time for 64
Search 7th time for 64
Search 8th time for 64
Search 9th time for 64
Search 10th time for 64
Search 11th time for 64
Search 12th time for 64
Search 13th time for 64
Search 14th time for 64
Search 15th time for 64
Search 16th time for 64
Search 17th time for 64
Search 18th time for 64
Search 19th time for 64
Search 20th time for 64
Search 21th time for 64
Search 22th time for 64
Search 23th time for 64
Search 24th time for 64
===Start search fo the 128 value===
Search 0th time for 128
Search 1th time for 128
Search 2th time for 128
Search 3th time for 128
Search 4th time for 128
Search 5th time for 128
Search 6th time for 128
Search 7th time for 128
Search 8th time for 128
Search 9th time for 128
Search 10th time for 128
Search 11th time for 128


KeyboardInterrupt: 

In [33]:
performance_results

{'movie_recommedation-memory_optimized': {64: {'avg_time': np.float64(225.3077507019043),
   'min_time': np.float64(202.23474502563477),
   'max_time': np.float64(297.27625846862793)},
  128: {'avg_time': np.float64(216.22923851013184),
   'min_time': np.float64(203.30286026000977),
   'max_time': np.float64(294.92807388305664)},
  256: {'avg_time': np.float64(213.93418312072754),
   'min_time': np.float64(203.97114753723145),
   'max_time': np.float64(257.75790214538574)}},
 'movie_recommedation-balanced': {64: {'avg_time': np.float64(211.9762420654297),
   'min_time': np.float64(199.44310188293457),
   'max_time': np.float64(220.4911708831787)},
  128: {'avg_time': np.float64(216.88876152038574),
   'min_time': np.float64(203.86695861816406),
   'max_time': np.float64(279.569149017334)},
  256: {'avg_time': np.float64(213.31461906433105),
   'min_time': np.float64(203.05824279785156),
   'max_time': np.float64(230.9722900390625)}},
 'movie_recommedation-high_quality': {64: {'avg_time

### Preliminary summary: probably 5000 points is not enough to use hnsw to take place. Increase more

In [34]:
from movie_recommendation import PointBuilder
from pathlib import Path
import os

# Set up paths for saving
cache_dir = Path("data/movie_vectors")
cache_dir.mkdir(parents=True, exist_ok=True)

# Check if we have cached points
cached_file = cache_dir / "movie_points_1000.pkl"
incremental_file = cache_dir / "movie_points_incremental.pkl"

point_builder = PointBuilder()

# Try to load existing points first
if cached_file.exists():
    print(f"Loading cached points from {cached_file}")
    movie_points = point_builder.load_points_from_disk(cached_file)
else:
    print("Building points from movies...")
    movie_points = []
    
    # Process in batches to enable incremental saving
    batch_size = 50  # Save every 50 movies
    total_movies = len(movies)
    
    for i in range(150, total_movies, batch_size):
        batch_end = min(i + batch_size, total_movies)
        batch_movies = movies[i:batch_end]
        
        print(f"\nProcessing movies {i+1}-{batch_end} of {total_movies}")
        
        # Build points for this batch
        batch_points = point_builder.build_points_from_movies(batch_movies)
        movie_points.extend(batch_points)
        
        # Save incrementally
        print(f"Saving {len(movie_points)} points incrementally...")
        point_builder.save_points_to_disk(movie_points, incremental_file)
        
        # Also save a summary for debugging
        summary_file = cache_dir / f"movie_points_summary_{batch_end}.json"
        point_builder.save_points_as_json(movie_points, summary_file)
        
        print(f"✓ Saved up to movie {batch_end}/{total_movies}")
    
    # Final save with proper filename
    print(f"\nSaving final {len(movie_points)} points to {cached_file}")
    point_builder.save_points_to_disk(movie_points, cached_file)
    
    # Clean up incremental file
    if incremental_file.exists():
        incremental_file.unlink()
        print("Cleaned up incremental file")

print(f"\nTotal points: {len(movie_points)}")
movie_points[:10]

Building points from movies...

Processing movies 151-200 of 1000
Building points for 50 movies with batch_size=32, max_workers=4
  Processed 50/50 images
Successfully encoded 50 images
Encoding text chunks in batches...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14046.47it/s]


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7073.72it/s]


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10611.97it/s]


  Encoding 250 fixed chunks in batches...
  Encoding 760 semantic chunks in batches...
Created 1371 points total
Saving 1371 points incrementally...
Saving 1371 points to data/movie_vectors/movie_points_incremental.pkl
Successfully saved points to data/movie_vectors/movie_points_incremental.pkl
Saving 1371 points to data/movie_vectors/movie_points_summary_200.json as JSON
Successfully saved points summary to data/movie_vectors/movie_points_summary_200.json
✓ Saved up to movie 200/1000

Processing movies 201-250 of 1000
Building points for 50 movies with batch_size=32, max_workers=4
  Processed 50/50 images
Successfully encoded 50 images
Encoding text chunks in batches...


...

  Encoding 216 fixed chunks in batches...
  Encoding 650 semantic chunks in batches...
Created 1160 points total
Saving 9454 points incrementally...
Saving 9454 points to data/movie_vectors/movie_points_incremental.pkl
Successfully saved points to data/movie_vectors/movie_points_incremental.pkl
Saving 9454 points to data/movie_vectors/movie_points_summary_550.json as JSON
Successfully saved points summary to data/movie_vectors/movie_points_summary_550.json
✓ Saved up to movie 550/1000

Processing movies 551-600 of 1000
Building points for 50 movies with batch_size=32, max_workers=4
  Processed 50/50 images
Successfully encoded 50 images
Encoding text chunks in batches...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12715.62it/s]


  Encoding 286 fixed chunks in batches...
  Encoding 834 semantic chunks in batches...
Created 1517 points total
Saving 10971 points incrementally...
Saving 10971 points to data/movie_vectors/movie_points_incremental.pkl
Successfully saved points to data/movie_vectors/movie_points_incremental.pkl
Saving 10971 points to data/movie_vectors/movie_points_summary_600.json as JSON
Successfully saved points summary to data/movie_vectors/movie_points_summary_600.json
✓ Saved up to movie 600/1000

Processing movies 601-650 of 1000
Building points for 50 movies with batch_size=32, max_workers=4
  Processed 50/50 images
Successfully encoded 50 images
Encoding text chunks in batches...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5824.79it/s]


KeyboardInterrupt: 

In [41]:
test_query = 'american movie'

performance_results_tenk = {}

for config in configs:
    if config.connection_per_node > 0:
        collection_name = f'movie_recommedation-{config.name}'
        performance_results_tenk[collection_name] = (
            collections[collection_name].benchmark_search(test_query)
        )

Collection info: m=100 ef_construct=100 full_scan_threshold=10 max_indexing_threads=0 on_disk=False payload_m=None inline_storage=None
Total points: 13916
Indexed vectors: 24230

===Start search for hnsw_ef=64===
Search 1/25: 189.40ms (found 10 results)
Search 6/25: 188.16ms (found 10 results)
Search 11/25: 191.33ms (found 10 results)
Search 16/25: 199.66ms (found 10 results)
Search 21/25: 190.02ms (found 10 results)
hnsw_ef=64: avg=190.72ms ±2.49ms

===Start search for hnsw_ef=128===
Search 1/25: 187.22ms (found 10 results)
Search 6/25: 189.62ms (found 10 results)
Search 11/25: 188.72ms (found 10 results)
Search 16/25: 190.91ms (found 10 results)
Search 21/25: 193.07ms (found 10 results)
hnsw_ef=128: avg=190.64ms ±3.76ms

===Start search for hnsw_ef=256===
Search 1/25: 248.03ms (found 10 results)
Search 6/25: 190.87ms (found 10 results)
Search 11/25: 203.43ms (found 10 results)
Search 16/25: 190.79ms (found 10 results)
Search 21/25: 190.62ms (found 10 results)
hnsw_ef=256: avg=200.65m

In [42]:
performance_results_tenk

{'movie_recommedation-memory_optimized': {64: {'avg_time': np.float64(190.7219409942627),
   'min_time': np.float64(187.5758171081543),
   'max_time': np.float64(199.6610164642334),
   'std_time': np.float64(2.491976206447863)},
  128: {'avg_time': np.float64(190.63517570495605),
   'min_time': np.float64(187.21985816955566),
   'max_time': np.float64(206.88986778259277),
   'std_time': np.float64(3.760079487757551)},
  256: {'avg_time': np.float64(200.6545352935791),
   'min_time': np.float64(189.37206268310547),
   'max_time': np.float64(345.8240032196045),
   'std_time': np.float64(31.699086408671125)}},
 'movie_recommedation-balanced': {64: {'avg_time': np.float64(193.07198524475098),
   'min_time': np.float64(187.6659393310547),
   'max_time': np.float64(243.70694160461426),
   'std_time': np.float64(10.566206203227862)},
  128: {'avg_time': np.float64(191.73606872558594),
   'min_time': np.float64(188.79294395446777),
   'max_time': np.float64(197.06392288208008),
   'std_time': 

In [43]:
collections['movie_recommedation-memory_optimized'].benchmark_brute_force_vs_hnsw('american movie')


=== Brute Force vs HNSW Comparison ===
Brute force 1/10: 778.22ms
Brute force 3/10: 189.05ms
Brute force 5/10: 190.47ms
Brute force 7/10: 190.66ms
Brute force 9/10: 192.74ms
HNSW 1/10: 188.21ms
HNSW 3/10: 192.74ms
HNSW 5/10: 189.46ms
HNSW 7/10: 186.91ms
HNSW 9/10: 196.63ms

Results:
Brute Force: 249.72ms ±176.17ms
HNSW:        189.91ms ±2.80ms
Speedup:     1.31x


{'brute_force': {'avg': np.float64(249.71933364868164),
  'std': np.float64(176.1741803118142)},
 'hnsw': {'avg': np.float64(189.91093635559082),
  'std': np.float64(2.800828229971277)},
 'speedup': np.float64(1.3149286630923933)}

## The reason why the performance did not change: queries are cached

In [53]:

test_query = ['american movie', 'phyco going after people', 'mario game', 'movie born from the meme']

performance_results_tenk = {}

for config in configs:
    if config.connection_per_node > 0:
        collection_name = f'movie_recommedation-{config.name}'
        performance_results_tenk[collection_name] = (
            collections[collection_name].benchmark_search(test_query)
        )

Collection info: m=100 ef_construct=100 full_scan_threshold=10 max_indexing_threads=0 on_disk=False payload_m=None inline_storage=None
Total points: 13916
Indexed vectors: 24230

===Start search for hnsw_ef=64===
Search 1/25: 193.93ms (found 10 results)
Search 6/25: 188.44ms (found 10 results)
Search 11/25: 184.21ms (found 10 results)
Search 16/25: 183.75ms (found 10 results)
Search 21/25: 184.16ms (found 10 results)
hnsw_ef=64: avg=184.94ms ±2.61ms

===Start search for hnsw_ef=128===
Search 1/25: 187.57ms (found 10 results)
Search 6/25: 190.64ms (found 10 results)
Search 11/25: 187.07ms (found 10 results)
Search 16/25: 185.38ms (found 10 results)
Search 21/25: 186.47ms (found 10 results)
hnsw_ef=128: avg=185.40ms ±1.64ms

===Start search for hnsw_ef=256===
Search 1/25: 188.63ms (found 10 results)
Search 6/25: 192.10ms (found 10 results)
Search 11/25: 184.71ms (found 10 results)
Search 16/25: 184.73ms (found 10 results)
Search 21/25: 184.47ms (found 10 results)
hnsw_ef=256: avg=185.91m

In [54]:
performance_results_tenk

{'movie_recommedation-memory_optimized': {64: {'avg_time': np.float64(184.9350643157959),
   'min_time': np.float64(182.5852394104004),
   'max_time': np.float64(193.92776489257812),
   'std_time': np.float64(2.612414609382294)},
  128: {'avg_time': np.float64(185.3999137878418),
   'min_time': np.float64(183.28619003295898),
   'max_time': np.float64(190.63878059387207),
   'std_time': np.float64(1.6386450254428737)},
  256: {'avg_time': np.float64(185.90606689453125),
   'min_time': np.float64(183.12501907348633),
   'max_time': np.float64(192.53778457641602),
   'std_time': np.float64(2.41524730438538)}},
 'movie_recommedation-balanced': {64: {'avg_time': np.float64(199.69146728515625),
   'min_time': np.float64(187.09897994995117),
   'max_time': np.float64(320.67084312438965),
   'std_time': np.float64(26.88672424663226)},
  128: {'avg_time': np.float64(195.61558723449707),
   'min_time': np.float64(187.22224235534668),
   'max_time': np.float64(233.66212844848633),
   'std_time':

In [47]:
movies[3].popularity

269.0824

In [56]:
collections['movie_recommedation-high_quality'].test_hnsw_vs_exact('american movie')


=== Testing HNSW vs Exact Search ===
Collection: movie_recommedation-high_quality
Points: 13916
Indexed vectors: 24230
HNSW m: 400
HNSW ef_construct: 400
Full scan threshold: 10

Testing exact search...
Testing HNSW search...

Results:
Exact search: 243.08ms ±66.80
HNSW search:  216.63ms ±30.78
Speedup:      1.12x
Result overlap: 8/10 (80.0%)

✅ HNSW appears to be working: 1.12x speedup


{'exact_avg': np.float64(243.07708740234375),
 'hnsw_avg': np.float64(216.62635803222656),
 'speedup': np.float64(1.1221030054255088),
 'overlap_percent': 80.0,
 'points_count': 13916,
 'full_scan_threshold': 10}

In [ ]:
selected_collection = 'movie_recommedation-balanced'


In [61]:
collections[selected_collection].client.delete_payload_index(collections[selected_collection].collection_name, 'popularity')
collections[selected_collection].client.delete_payload_index(collections[selected_collection].collection_name, 'vote_average')

UpdateResult(operation_id=289, status=<UpdateStatus.COMPLETED: 'completed'>)

In [62]:
filtered_results = collections[selected_collection].benchmark_with_filtering('best american moview in history')

===Start search with filtering===
Time spend on query 221.41289710998535 for the cycle 0/25
Time spend on query 222.21899032592773 for the cycle 1/25
Time spend on query 217.71788597106934 for the cycle 2/25
Time spend on query 219.12884712219238 for the cycle 3/25
Time spend on query 231.21118545532227 for the cycle 4/25
Time spend on query 224.168062210083 for the cycle 5/25
Time spend on query 220.36099433898926 for the cycle 6/25
Time spend on query 222.8548526763916 for the cycle 7/25
Time spend on query 217.96822547912598 for the cycle 8/25
Time spend on query 224.74002838134766 for the cycle 9/25
Time spend on query 221.38595581054688 for the cycle 10/25
Time spend on query 221.49300575256348 for the cycle 11/25
Time spend on query 222.36394882202148 for the cycle 12/25
Time spend on query 224.7469425201416 for the cycle 13/25
Time spend on query 240.69666862487793 for the cycle 14/25
Time spend on query 268.0661678314209 for the cycle 15/25
Time spend on query 228.8730144500732

In [65]:
performance_results_tenk

{'movie_recommedation-memory_optimized': {64: {'avg_time': np.float64(184.9350643157959),
   'min_time': np.float64(182.5852394104004),
   'max_time': np.float64(193.92776489257812),
   'std_time': np.float64(2.612414609382294)},
  128: {'avg_time': np.float64(185.3999137878418),
   'min_time': np.float64(183.28619003295898),
   'max_time': np.float64(190.63878059387207),
   'std_time': np.float64(1.6386450254428737)},
  256: {'avg_time': np.float64(185.90606689453125),
   'min_time': np.float64(183.12501907348633),
   'max_time': np.float64(192.53778457641602),
   'std_time': np.float64(2.41524730438538)}},
 'movie_recommedation-balanced': {64: {'avg_time': np.float64(199.69146728515625),
   'min_time': np.float64(187.09897994995117),
   'max_time': np.float64(320.67084312438965),
   'std_time': np.float64(26.88672424663226)},
  128: {'avg_time': np.float64(195.61558723449707),
   'min_time': np.float64(187.22224235534668),
   'max_time': np.float64(233.66212844848633),
   'std_time':

In [66]:
print("=" * 60)
print("PERFORMANCE OPTIMIZATION RESULTS")
print("=" * 60)


print("\n2) Search Performance (hnsw_ef=128):")
for config_name, results in performance_results_tenk.items():
    if 128 in results:
        print(f"   {config_name}: {results[128]['avg_time']:.2f}ms")

print("\n3) Filtering Impact:")
print(f"   Without index: {filtered_results['without_index']:.2f}ms")
print(f"   With index: {filtered_results['with_index']:.2f}ms")
print(f"   Speedup: {filtered_results['speedup']:.1f}x")

PERFORMANCE OPTIMIZATION RESULTS

2) Search Performance (hnsw_ef=128):
   movie_recommedation-memory_optimized: 185.40ms
   movie_recommedation-balanced: 195.62ms
   movie_recommedation-high_quality: 192.95ms

3) Filtering Impact:
   Without index: 226.28ms
   With index: 194.44ms
   Speedup: 1.2x



[Day 2] HNSW Performance Benchmarking

  High-Level Summary
  - Domain: "Movie Recommendation System (TMDB + Wikipedia plots)"
  - Key Result: "m=8, ef_construct=100, hnsw_ef=128 gave 185 ms search and 161 s upload (best balance)."

  The Journey
  Started with 100 movies expecting to see HNSW benefits, but performance was flat across all configurations (344-391ms). Discovered this was below HNSW's effectiveness 
  threshold. Scaled to 300 movies (→5,000 chunks) - still no improvement (211-225ms). Finally used ~600 movies which generated 13,916 chunks after text segmentation where
  modest HNSW benefits emerged. The preprocessing bottleneck was significant: fetching Wikipedia plots + encoding text/images meant hours of preparation for each test
  iteration.

  Reproducibility
  - Collections: movie_recommendation-{fast_initial_upload, memory_optimized, balanced, high_quality}
  - Model: sentence-transformers/all-MiniLM-L6-v2 (384-dim) for text, clip-ViT-B-32 (512-dim) for posters
  - Dataset: ~600 movies → 13,916 chunks with 3 chunking strategies (snapshot: 2025-01-06)

  Configuration Results
  | m   | ef_construct | Upload_s | Search_ms@ef=128 |
  |-----|--------------|----------|------------------|
  | 0   | 100          | —        | —                |
  | 8   | 100          | 160.8    | 185.4            |
  | 16  | 200          | 168.1    | 195.6            |
  | 32  | 400          | 193.5    | 193.0            |

  Filtering Impact
  - Payload index on popularity & vote_average: 1.2×Without index: 226.3 ms → With index: 194.4 ms

  Recommendations
  - Best config for this domain: [m=8, ef_construct=100, hnsw_ef=128]
  - When to pick another setting: Use m=0 for <10K vectors, m=32 only for >100K vectors where quality matters
  - Notes for production: Always index filter fields; network latency (~180ms) dominates at this scale

  Surprise
  - "Query caching made initial benchmarks misleading - first query took 778ms, subsequent ones 190ms regardless of algorithm"

  What Wasn't Tested
  - Multimodal search combining text + image vectors (30/70 weighting)
  - Impact of different chunking strategies on search quality
  - Local Qdrant to isolate true algorithm performance from network overhead

  Next Step
  - "Deploy local Qdrant instance to measure sub-millisecond HNSW performance and test multimodal search latency"
